# Lab 3\. Enabling Automatic Mixed Precision in a Training Loop

## Overview

In the chapter you saw that not every operation in a training step needs full 32-bit precision, and that Automatic Mixed Precision (AMP) automates the choice: it runs the tolerant operations, such as convolutions and linear layers, in a lower-precision 16-bit format, while keeping numerically sensitive operations, such as the loss, in `float32`. You also read the five lines AMP adds to a training loop. This lab is where you make those five edits yourself.

You will start from a plain `float32` training loop for a small image classifier and turn it into an AMP-enabled loop, one edit at a time. You will create a `GradScaler`, wrap the forward pass and loss in an `autocast` context, and convert the backward and optimizer lines to their scaler-aware forms. After each idea is in place, you will run the finished loop, confirm the model still trains, and inspect the data types to see the mixed-precision split with your own eyes.

The practical payoff is the ability to recognize and apply the AMP pattern in real training code. AMP is one of the first tools an ML practitioner reaches for when a model is close to filling GPU memory or when training time matters on a CUDA GPU. The exercises here run on CPU so you can follow the mechanics anywhere, which also makes the honest limitation concrete: on CPU the code runs correctly and the model trains, but the speed benefit is minimal, and you will see why.

## Learning Objectives

By the end of this lab, you should be able to:

- Enable Automatic Mixed Precision on an existing PyTorch training loop using `torch.amp.autocast` and `torch.amp.GradScaler`  
- Identify and explain what each of the five AMP-related lines does in the loop  
- Run an AMP-enabled training loop and confirm the model still trains  
- Inspect operation data types to observe which parts of the step run in lower precision and which stay in `float32`  
- Explain from direct observation why AMP's speed benefit appears on a CUDA GPU rather than on CPU

## Prerequisites

This course’s labs are **not cumulative**, so this lab does not depend on any prior resources or the state of earlier labs if there are any.

You need the following before you start:

- **A notebook environment:** either a Google account for Google Colab, or a local installation of Jupyter Notebook. Every step in this lab runs on a CPU; no GPU is required.  
- **Python 3.12 or later.** On Colab this is already provided. For a local environment, confirm your version with `python --version`.  
- **PyTorch 2.12.1,** installed with `pip` in the previous lab, which provides the unified `torch.amp` namespace used throughout this lab. If you are on Colab, these may already be present.  
- **A web browser** (Chrome, Firefox, or Edge are recommended for Colab).  
-

>   
> **Note:** No GPU is required. The lab detects your hardware and runs on CPU by default. If you happen to have a CUDA GPU available (for example, by enabling the Colab GPU runtime), the same code will use it, and the timing comparison near the end will be more interesting.

## Estimated Time

15–20 minutes.

## How to Perform the Exercise

You will work entirely in one notebook, running cells from top to bottom. Create a new notebook in Google Colab (**File \> New notebook**) or a new `.ipynb` in local Jupyter, and add each code block below as its own cell as you reach it. Download the chapter 3 lab notebook from the [LFD2003-resources GitHub repository](https://github.com/lftraining/LFD2003-resources.git) from the **LFD2003-resources/labs** directory. In Google Colab, select File, then Upload Notebook, and select the chapter 3 lab notebook you downloaded.

Read the explanation before running each cell, run it, and compare what you see to the Sample Output or Output shown after each block. Sample Output means the exact values will differ from run to run (random tensors, dataset counts, file paths), so match the shape and structure rather than every character. Output marks a checkpoint where a specific outcome confirms the step worked.

Commands in this lab use the notebook form, where a line beginning with `!` runs a shell command from inside a notebook cell (for example, `!pip install ...`). If you run these in a local terminal instead of a notebook cell, drop the leading `!`.

> **Note:** Run the cells in order and only once each, unless a step says otherwise. If you restart the notebook's runtime, re-run the cells from the top, because the variables live in memory and are cleared on restart.

## Setup: Environment Preparation

The steps in this section stand up what the later sections need: the imports, a synthetic dataset, and the model from Lab 2\. They are not the learning content of the lab, and on a managed course platform some of them may already be done for you. Run them once at the top of your notebook, then move on to Part A.

### Setup 1\. Import PyTorch and Select the Device

Import the libraries and detect the hardware. The device you are on determines both where the tensors live and which lower-precision type AMP will use, so you resolve it once here and reuse it everywhere.


In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(0)  # makes the run reproducible

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


**Sample Output**

```
Using device: cpu
```

The device will read `cuda` if a CUDA GPU is available and `cpu` otherwise. Most learners will see `cpu`, which is expected and fine for this lab.

### Setup 2\. Define a Small Classifier

Define the model you will train: a small convolutional network that takes a 28-by-28 grayscale image and predicts one of 10 classes. This is the same kind of small classifier the chapter used as its running example. The architecture itself is not the point of the lab; it just gives AMP some convolutions and linear layers to work on.


In [ ]:
class SmallClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x




### Setup 3\. Create Synthetic Data

Create a fixed set of random input batches and random labels, plus the loss function and optimizer you will reuse in both loops. Synthetic data keeps the lab self-contained with no download, and reusing the same fixed batches across epochs lets the model fit them, so you can watch the loss trend downward as evidence that training is working.


In [ ]:
num_batches = 50
inputs = [torch.randn(64, 1, 28, 28, device=device) for _ in range(num_batches)]
targets = [torch.randint(0, 10, (64,), device=device) for _ in range(num_batches)]




Each entry in `inputs` is a batch of 64 grayscale images, and each entry in `targets` is the 64 matching class labels. `CrossEntropyLoss` is the standard classification loss, and it is one of the operations AMP keeps in `float32` because it needs the wider range.

## Part A: Establish the Baseline (float32)

Before enabling AMP, run the ordinary loop so you have a working reference and a baseline time to compare against.

### Step 1\. Write and Run the Baseline Training Loop

Train the model in default precision. Every operation here runs in `float32`. The loop does the four familiar things per batch: forward pass, loss, backward pass, and optimizer step. The timing calls around the loop give you a number to compare the AMP version against.


In [ ]:
model = SmallClassifier().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()
epochs = 5

start = time.time()
for epoch in range(epochs):
    for x, y in zip(inputs, targets):
        output = model(x)                 # forward pass
        loss = loss_fn(output, y)         # compute loss
        loss.backward()                   # backpropagate
        optimizer.step()                  # update weights
        optimizer.zero_grad()             # clear gradients
    print(f"[baseline] epoch {epoch + 1}  loss {loss.item():.4f}")

print(f"[baseline] total time: {time.time() - start:.3f} s")


**Sample Output (CPU)**

```
[baseline] epoch 1  loss 2.3104
[baseline] epoch 2  loss 2.3075
[baseline] epoch 3  loss 2.3056
[baseline] epoch 4  loss 2.3041
[baseline] epoch 5  loss 2.3009
[baseline] total time: 6.613 s
```

**Sample Output (GPU)**

```
...
[baseline] total time: 1.563 s
```

Your loss values and time will differ, but the loss should trend downward across epochs. That downward trend confirms the training loop is optimizing the model, which is the behavior you want to preserve after enabling AMP.

## Part B: Enable Automatic Mixed Precision

You will now make the five edits that turn the baseline into an AMP loop. Each edit is one step. Read the change and what it does, and note the exact line involved; you will assemble all five into a complete, runnable loop in Step 10\.

Two of the edits depend on your hardware, so set the two AMP settings once, right after Part A. On a CUDA GPU, AMP uses `float16` and the `GradScaler` is active. On CPU, AMP uses `bfloat16`, whose wider range means gradient scaling is generally unnecessary, so the scaler is turned off. The chapter showed the canonical CUDA form `torch.amp.autocast(device_type="cuda", dtype=torch.float16)`; here you make it adapt to the device so the same loop runs correctly on your CPU while still exercising all five edits.


In [ ]:
device_type = torch.device(device).type
autocast_dtype = torch.float16 if device_type == "cuda" else torch.bfloat16
use_scaler = (autocast_dtype == torch.float16)
print("autocast dtype:", autocast_dtype, "| GradScaler active:", use_scaler)


**Sample Output**

```
autocast dtype: torch.bfloat16 | GradScaler active: False
```

### Step 2\. Create the GradScaler (edit 1 of 5\)

Create one `GradScaler` before the loop begins. You use a single scaler for the whole run so it can tune its scale factor over time. The `enabled` argument makes the scaler a no-op when it is off, which is why the same code is correct on CPU (`bfloat16`, no scaling needed) and on GPU (`float16`, scaling active).


In [ ]:
scaler = torch.amp.GradScaler(device, enabled=use_scaler)


### Step 3\. Wrap the Forward Pass and Loss in autocast (edit 2 of 5\)

Place the forward pass and the loss computation inside a `torch.amp.autocast` context. Inside this context, PyTorch runs each operation in the precision it deems appropriate: the convolutions and linear layers in the lower-precision type, and the loss in `float32`. Only the forward pass and loss go inside; you exit the context before the backward pass.


In [ ]:
with torch.amp.autocast(device_type=device, dtype=autocast_dtype):
    output = model(x)
    loss = loss_fn(output, y)

print(loss)


**Sample Output**

```
tensor(2.2905, device='cuda:0', grad_fn=<NllLossBackward0>)
```

### Step 4\. Scale the Loss, Then Backpropagate (edit 3 of 5\)

Replace `loss.backward()` with `scaler.scale(loss).backward()`. This multiplies the loss by the scale factor before backpropagation, so the small gradients produced in `float16` are lifted into a range the format can represent instead of underflowing to zero. When the scaler is off (CPU), this behaves like an ordinary `backward()`.


In [ ]:
# before:  loss.backward()

scaled_loss = scaler.scale(loss)
scaled_loss.backward()

print(scaled_loss)


**Sample Output (CPU)**

```
tensor(2.2905, device='cuda:0', grad_fn=<NllLossBackward0>)
```

**Sample Output (GPU)**

```
tensor(148812., device='cuda:0', grad_fn=<MulBackward0>)
```

### Step 5\. Step the Optimizer Through the Scaler (edit 4 of 5\)

Replace `optimizer.step()` with `scaler.step(optimizer)`. The scaler first unscales the gradients back to their true size, then checks them for infinities or invalid values, and applies the weight update only if the gradients are valid. A step with invalid gradients is skipped rather than corrupting the model.


In [ ]:
# before:  optimizer.step()
scaler.step(optimizer)


### Step 6\. Update the Scale Factor (edit 5 of 5\)

Add `scaler.update()` after the optimizer step. This adjusts the scale factor for the next iteration, raising it when steps succeed and lowering it when a step had to be skipped, so the scaling stays in a useful range as training proceeds.


In [ ]:
scaler.update()


### Step 7\. Assemble and Run the Full AMP Loop

Put the five edits together into a complete loop and run it. A fresh model and optimizer make this a clean, comparable run against the baseline. Watch that the loss still trends downward, which confirms the AMP-enabled loop trains the model correctly.


In [ ]:
model = SmallClassifier().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
scaler = torch.amp.GradScaler(device, enabled=use_scaler)   # edit 1

start = time.time()
for epoch in range(epochs):
    for x, y in zip(inputs, targets):
        with torch.amp.autocast(device_type=device, dtype=autocast_dtype):  # edit 2
            output = model(x)
            loss = loss_fn(output, y)
        scaler.scale(loss).backward()     # edit 3
        scaler.step(optimizer)            # edit 4
        scaler.update()                   # edit 5
        optimizer.zero_grad()
    print(f"[amp] epoch {epoch + 1}  loss {loss.item():.4f}")

print(f"[amp] total time: {time.time() - start:.3f} s")


**Sample Output (CPU)**

```
[amp] epoch 1  loss 2.3107
[amp] epoch 2  loss 2.3057
[amp] epoch 3  loss 2.3013
[amp] epoch 4  loss 2.2975
[amp] epoch 5  loss 2.2943
[amp] total time: 37.601 s
```

**Sample Output (GPU)**

```
...
[amp] total time: 0.584 s
```

**Expected Result:** the loop runs without error and the loss trends downward across the five epochs, just as the baseline did. The model trains the same way; only the precision of the operations changed.

> **Note:** On CPU, AMP may take about the same time as the baseline and can sometimes be slower. Unlike CUDA GPUs, not every CPU has efficient reduced-precision hardware, and small workloads may not recover the overhead introduced by autocasting. Modern CPUs with native BF16 support can still benefit substantially.  
> On a Colab runtime, BF16 autocast is likely to add more overhead than it saves, probably because the small workload and assigned CPU do not provide an effective native BF16 support (it has emulated support, though). In Colab or any Linux machine, the hardware-accelerated support can be checked using the snippet below.


In [ ]:
with open("/proc/cpuinfo") as f:
    cpuinfo = f.read().lower()

for feature in ["avx512_bf16", "amx_bf16"]:
    print(f"{feature:14s}:", feature in cpuinfo)


**Sample Output**

```
avx512_bf16   : False
amx_bf16      : False
```

>

## Part C: Observe What AMP Did

The loop trained, but you have not yet seen the mixed-precision split directly. This short section makes it visible and interprets the timing you saw.

### Step 8\. Inspect the Operation Data Types

Run a single forward pass inside the `autocast` context and print the data types of the model output and the loss. This shows the split the chapter described: the model's output comes out in the lower-precision type, while the loss stays in `float32`.


In [ ]:
with torch.amp.autocast(device_type=device, dtype=autocast_dtype):
    output = model(inputs[0])
    loss = loss_fn(output, targets[0])

print("output dtype inside autocast:", output.dtype)
print("loss dtype inside autocast:  ", loss.dtype)


**Sample Output**

```
output dtype inside autocast: torch.bfloat16
loss dtype inside autocast:   torch.float32
```

The model output is in the lower-precision type that `autocast` selected (`torch.bfloat16` on CPU, or `torch.float16` on a CUDA GPU), because the convolutions and linear layers that produced it are the tolerant operations AMP runs in low precision. The loss is `torch.float32` because `CrossEntropyLoss` is a numerically sensitive operation that `autocast` keeps in full precision. This is exactly what "mixed" precision means: two precisions inside one step, chosen per operation.

### Step 9\. Interpret the Timing

Compare the baseline time from Step 4 with the AMP time from Step 10\. On a CPU, the AMP version takes much longer because Colab’s CPU does not provide native (hardware-accelerated) support for the BF16 type. The BF16 emulation can add substantial overhead, thus making execution actually slower than when using the FP32 type. On a CUDA GPU with Tensor Cores, the AMP loop would be meaningfully faster (almost 3x in our example), and it would use less memory, which is where AMP earns its place. Interpreting your own numbers this way is the diagnostic habit the chapter set up: the benefit is a hardware question, not a property of the code.

## Cleanup

This lab does not create anything that must be torn down, and part of what it produces may be kept. There are no processes to stop and no services to remove.

On Google Colab the runtime is temporary. When your Colab session ends, all loaded objects are cleared.

## Summary

You took a working `float32` training loop and enabled Automatic Mixed Precision with five edits: you created a `GradScaler`, wrapped the forward pass and loss in `torch.amp.autocast`, and switched to `scaler.scale(loss).backward()`, `scaler.step(optimizer)`, and `scaler.update()`. Running the finished loop showed the model still trains, and printing the data types showed the mixed-precision split the chapter described, with the model output in a 16-bit type and the loss in `float32`. Comparing the baseline and AMP times made the hardware dependence concrete: on CPU there is little or no speedup, while the real gains in speed and memory appear on a CUDA GPU. The notebook created no lasting files or resources, so there is nothing to carry into the next chapter, where you will move from applying optimizations to measuring where a training loop actually spends its time.  
